# Analytical Benchmark Case

This notebook runs one compact Low & Lou analytical field benchmark. It is meant as a fast release smoke test: train against the full analytical boundary, export the result, and compare the reconstructed field against the known solution.

Use `CASE = 1` for the quickest check, or `CASE = 2` for a slightly larger analytical volume.

## 1. Select The Benchmark

Choose the analytical case, output directory, height range, and training length. The defaults keep the run small enough for a notebook while still exercising the full Cartesian training and export path.

In [ ]:
from pathlib import Path

RUN_TRAINING = True
CASE = 1
RUN_ROOT = Path("runs/benchmark")
EXPORT_MM_PER_PIXEL = 0.72
HEIGHT_MIN = 0
HEIGHT_MAX = 2
EPOCHS = 15
VALIDATION_PIXEL_PER_DS = 32
CASE_RESOLUTION = {1: 64, 2: [80, 80, 72]}

## 2. Import Packages

The W&B console capture is disabled for notebooks to avoid Rich/IPython output recursion while keeping W&B metric and image logging enabled.

In [ ]:
import os
os.environ.setdefault("WANDB_CONSOLE", "off")

from pathlib import Path
from dateutil.parser import parse

from astropy import units as u
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm

from nf2.data.analytical_field import get_analytic_b_field
from nf2.evaluation.metric import evaluate

## 3. Train The Analytical Model

The benchmark trains on all analytical boundary faces and validates only the bottom boundary plus slices. The learning rate is fixed at `1e-4` for the full training run.

In [ ]:
run_dir = RUN_ROOT / f"case{CASE}"
config = {
    "path": str(run_dir),
    "work_path": str(run_dir / "work"),
    "logging": {"project": "nf2", "name": f"Analytical case {CASE}"},
    "model": {"network": {"type": "siren", "w0_init": 1}},
    "data": {
        "geometry": "cartesian",
        "normalization": {"Mm_per_ds": 1, "Gauss_per_dB": 1},
        "boundaries": [
            {
                "id": "boundary",
                "type": "analytical",
                "case": CASE,
                "boundary": "full",
                "resolution": CASE_RESOLUTION[CASE],
                "batch_size": 4096,
            }
        ],
        "sampler": {"type": "height", "batch_size": 8192},
        "potential_boundary": {"type": "none"},
        "validation": [
            {
                "id": "boundary",
                "type": "analytical",
                "case": CASE,
                "boundary": "bottom",
                "resolution": CASE_RESOLUTION[CASE],
                "batch_size": 4096,
            },
            {"id": "cube", "type": "cube", "batch_size": 8192},
            {"id": "slices", "type": "slices", "n_slices": 5, "batch_size": 8192},
        ],
        "iterations": 1000,
        "z_range": [HEIGHT_MIN, HEIGHT_MAX],
        "validation_pixel_per_ds": VALIDATION_PIXEL_PER_DS,
    },
    "training": {"epochs": EPOCHS, "optimizer": 1.0e-4},
    "losses": [
        {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
        {"type": "force_free", "name": "force_free", "weight": 1.0e-3, "datasets": ["random"]},
    ],
    "loss_scaling": [],
    "callbacks": [
        {"type": "boundary", "dataset": "boundary"},
        {"type": "metrics", "dataset": "cube"},
        {"type": "slices", "dataset": "slices"},
    ],
}
if RUN_TRAINING:
    nf2.run(**config)
    wandb.finish()
else:
    config
NF2_PATH = run_dir / "extrapolation_result.nf2"
NF2_PATH

## 4. Export And Score

Export the trained `.nf2` state to a cube, load the matching analytical reference field, and compute the comparison metrics.

In [ ]:
EXPORT_DIR = NF2_PATH.parent / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / f"case{CASE}.vtk"), fmt="vtk", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy_fft"])

out = nf2.load(NF2_PATH)
cube = out.load_cube(Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "free_energy_fft"], progress=True)

b = cube["b"].to_value(u.G)
j_current = cube["metrics"]["j"].to_value(u.G / u.s)
j = vector_norm(j_current)
free_energy = cube["metrics"]["free_energy_fft"].to_value(u.erg / u.cm**3)
voxel_volume_cm3 = (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm) ** 3
reference = get_analytic_b_field(n=1, m=1, l=0.3, psi=np.pi / 4 if CASE == 1 else np.pi * 0.15, resolution=list(b.shape[:3]), bounds=[-1, 1, -1, 1, 0, 2])
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_current)),
    "sigma_J_1e2": float(sigma_J(b, j_current) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(j)),
    "total_free_energy_erg": float(np.nansum(free_energy) * voxel_volume_cm3),
    **{f"reference_{k}": float(v) for k, v in evaluate(b, reference).items() if np.isscalar(v)},
}
metrics

## 5. Inspect The Result

These quick-look slices show the reconstructed field alongside the analytical target so failures are easy to spot before looking at the scalar metrics.

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "free_energy_fft"], progress=True)
b = cube["b"].to_value(u.G)
j_current = cube["metrics"]["j"].to_value(u.G / u.s)
j = vector_norm(j_current)
free_energy = cube["metrics"]["free_energy_fft"].to_value(u.erg / u.cm**3)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
current_map = np.nansum(j, axis=2) * EXPORT_MM_PER_PIXEL
free_energy_map = np.nansum(free_energy, axis=2) * (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm)
boundary_bz = b[:, :, 0, 2]
extent = [0, b.shape[0] * EXPORT_MM_PER_PIXEL, 0, b.shape[1] * EXPORT_MM_PER_PIXEL]

def log_norm(image, lower=1, upper=99):
    positive = image[np.isfinite(image) & (image > 0)]
    if positive.size == 0:
        return LogNorm(vmin=np.nextafter(0, 1), vmax=1.0)
    vmin, vmax = np.nanpercentile(positive, [lower, upper])
    return LogNorm(vmin=max(vmin, np.nextafter(0, 1)), vmax=max(vmax, vmin * 1.01))

im = axs[0].imshow(current_map.T, origin="lower", cmap="inferno", norm=log_norm(current_map), extent=extent)
axs[0].set_title("Height-integrated |J| (log)")
plt.colorbar(im, ax=axs[0], fraction=0.046, label="Height-integrated |J| [G Mm s$^{-1}$]")

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="jet", norm=log_norm(free_energy_map), extent=extent)
axs[1].set_title("Height-integrated free energy (log)")
plt.colorbar(im, ax=axs[1], fraction=0.046, label="Height-integrated free energy [erg cm$^{-2}$]")

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=extent)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046, label="$B_z$ [G]")

for ax in axs:
    ax.set_xlabel("x [Mm]")
    ax.set_ylabel("y [Mm]")
fig.tight_layout()
plt.show()